# 06_01 · Modelo Final — Capa 1 (SPARK, detección eléctrica)

**Promoción a producción** de los hiperparámetros afinados en `05_04_Modelos_SPARK`.

Hasta ahora los artefactos de producción (los parquet `spark_*_scored` y los modelos
por máquina) los generaba `03_04` con valores **por defecto** (`contamination=0.05`,
`n_estimators=200`). Este notebook:

1. Lee los hiperparámetros afinados de `models/spark_model.pkl` (si `05_04` se ha
   ejecutado; si no, usa los valores por defecto y lo avisa).
2. Entrena un Isolation Forest **por máquina** con esos hiperparámetros.
3. **Re-puntúa y sobrescribe** los parquet `spark_{m}_scored.parquet` y el consolidado.
4. Guarda el bundle de producción `models/spark_models.pkl` (modelos + scalers + params).

> ⚠️ Sobrescribe artefactos de producción: ejecutar solo cuando se decida promocionar.


In [1]:
import sys; sys.path.insert(0, '../../src')
import warnings; warnings.filterwarnings('ignore')
import os, pickle
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest

from paths import PROCESSED, MODELS
from spark import MACHINES, FEATURE_COLS

with open(os.path.join(PROCESSED, 'spark_scalers.pkl'), 'rb') as f:
    scalers = pickle.load(f)

# Hiperparámetros: afinados (05_04) si existen, defaults (03_04) si no
tuned_path = os.path.join(MODELS, 'spark_model.pkl')
if os.path.exists(tuned_path):
    with open(tuned_path, 'rb') as f:
        params = pickle.load(f)['best_params']
    print(f'Hiperparámetros AFINADOS (05_04): {params}')
else:
    params = {'n_estimators': 200, 'max_features': 1.0, 'contamination': 0.05}
    print(f'⚠ spark_model.pkl no encontrado — usando defaults de 03_04: {params}')


Hiperparámetros AFINADOS (05_04): {'n_estimators': 200, 'max_features': 0.75, 'contamination': 0.01}


## 1. Entrenar y re-puntuar por máquina


In [2]:
models_prod = {}
summary = []

for m in MACHINES:
    df_m = pd.read_parquet(os.path.join(PROCESSED, f'spark_{m}_1min.parquet'))
    cols = [c for c in FEATURE_COLS if c in df_m.columns]

    # Entrenar solo con minutos de operación
    mask_op = ~df_m['is_gap']
    df_op = df_m[mask_op][cols].dropna()
    X_op = scalers[m].transform(df_op)

    ifo = IsolationForest(n_estimators=int(params['n_estimators']),
                          max_features=float(params['max_features']),
                          contamination=float(params['contamination']),
                          random_state=42, n_jobs=-1)
    ifo.fit(X_op)
    models_prod[m] = ifo

    # Re-puntuar todos los registros (misma lógica que 03_04)
    df_m['anomaly_score'] = np.nan
    df_m['is_anomaly'] = -1   # -1 = gap

    idx_op = df_m[mask_op].dropna(subset=cols).index
    X_all = scalers[m].transform(df_m.loc[idx_op, cols])
    df_m.loc[idx_op, 'anomaly_score'] = ifo.decision_function(X_all)
    df_m.loc[idx_op, 'is_anomaly'] = (ifo.predict(X_all) == -1).astype(int)

    df_m.to_parquet(os.path.join(PROCESSED, f'spark_{m}_scored.parquet'))

    n_anom = int((df_m['is_anomaly'] == 1).sum())
    n_op = int((df_m['is_anomaly'] >= 0).sum())
    summary.append({'Máquina': m.replace('TEC_', ''), 'Min op': n_op,
                    'Anomalías': n_anom, '%': round(n_anom / n_op * 100, 2)})
    print(f'{m:<20} {n_anom:>6,} anomalías / {n_op:>7,} op ({n_anom/n_op:.1%})')

summary_df = pd.DataFrame(summary).set_index('Máquina')
print(f'\nDesviación de tasa entre máquinas: {summary_df["%"].std():.2f} pp')


TEC_48S                 159 anomalías /  15,802 op (1.0%)
TEC_CFST161           4,777 anomalías / 477,781 op (1.0%)
TEC_CTX800TC            658 anomalías /  65,787 op (1.0%)
TEC_Chiron800         1,046 anomalías / 104,596 op (1.0%)
TEC_DMF3008             892 anomalías /  89,196 op (1.0%)
TEC_DMU125MB            738 anomalías /  73,744 op (1.0%)
TEC_DNG50evo          4,778 anomalías / 477,786 op (1.0%)
TEC_E110                832 anomalías /  83,161 op (1.0%)
TEC_E30D2               852 anomalías /  85,165 op (1.0%)
TEC_JWA24             4,778 anomalías / 477,788 op (1.0%)
TEC_MV2400R           1,816 anomalías / 181,588 op (1.0%)

Desviación de tasa entre máquinas: 0.00 pp


## 2. Consolidar y guardar producción


In [3]:
# Consolidado global
all_scored = pd.concat([
    pd.read_parquet(os.path.join(PROCESSED, f'spark_{m}_scored.parquet'))
    for m in MACHINES
])
all_scored.to_parquet(os.path.join(PROCESSED, 'spark_tec_scored.parquet'))
print(f'spark_tec_scored.parquet actualizado: {all_scored.shape}')

# Bundle de producción Capa 1
bundle = {
    'models':       models_prod,    # IsolationForest por máquina
    'scalers':      scalers,        # StandardScaler por máquina
    'params':       params,
    'feature_cols': FEATURE_COLS,
    'summary':      summary_df,
}
out = os.path.join(MODELS, 'spark_models.pkl')
with open(out, 'wb') as f:
    pickle.dump(bundle, f)
print(f'Guardado: {out}')


spark_tec_scored.parquet actualizado: (5797440, 40)
Guardado: /Users/nora/Documentos_Clase/BOOTCAMP/Proyecto_ML/Parada_de_maquina/models/spark_models.pkl


## 3. Conclusiones

- Los modelos de producción de la Capa 1 quedan en `models/spark_models.pkl`
  (un Isolation Forest + scaler por máquina, con los hiperparámetros afinados).
- Los parquet `scored` quedan re-puntuados — `04_04` y `07` reflejarán el afinado
  en su próxima ejecución.
